In [1]:
import sys
import json 
import joblib
import gc
from tqdm import tqdm
import os
from typing import List, Dict, Tuple
import yaml

# TO CHANGE
BASEDIR = "../../../"
sys.path.insert(0, BASEDIR)

from src.pipelines.memorize import MemPipelineConfig, MemPipeline, LLMExtractorConfig, LLMUpdatorConfig
from src.kg_model import KnowledgeGraphModel, EmbeddingsModelConfig, GraphModelConfig, EmbedderModelConfig
from src.db_drivers.graph_driver import GraphDBConnectionConfig, GraphDriverConfig
from src.db_drivers.vector_driver import VectorDBConnectionConfig, VectorDriverConfig

# gigachat key
#GIGACHAT_CREDS = 'OWUwOGUzOWEtMjJiNi00YmMxLThmMmItNzMwNjM2MTI2YmYxOjg2ODdiOTVhLTZkNDctNGFjOC1iMmViLTEyNDA5MmFiN2Q5Mw=='
# openai key
#API_KEY = "'sk-861mINAavom2SSBqgrI82D4thMOfqT37knCof2o0H0T3BlbkFJ2gdVXJuVjNesNNP2aeUwPoBpZP3a3R1gn1kqv97CsA'"

gc.collect()

/home/dzigen/Desktop/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


64

### Setting knowledge-graph configuration

In [2]:
# Read YAML file
with open("params.yaml", 'r') as stream:
    HYPER_PARAMS = yaml.safe_load(stream)

In [3]:
BASE_PATH = "../../../data/knowledge_graphs/"
DATASET_PATH = BASE_PATH + f"{HYPER_PARAMS['DATASET_NAME']}/"
KG_PATH = DATASET_PATH + f"{HYPER_PARAMS['KNOWLEDGE_GRAPH_NAME']}/"

HYPER_PARAMS_PATH = KG_PATH + 'hyperparameters.json'
EXTRACTED_TRIPLETS_PATH = KG_PATH + "extracted_triplets"
GRAPH_DRIVER_CONFIG_PATH = KG_PATH + "graph_config"
EMBEDDINGS_DRIVER_CONFIG_PATH = KG_PATH + "embeddings_config"
MEM_PIPELINE_CONFIG_PATH = KG_PATH + "mem_pipeline_config"

VECTORIZED_DB_PATH = KG_PATH + "embeddings_part/"
GRAPH_DB_PATH = KG_PATH + "graph_part/"
CACHE_DB_PATH = KG_PATH + "cache_part/"

In [ ]:
if not os.path.exists(BASE_PATH):
    raise ValueError(f"Директории не существует: {BASE_PATH}")
if not os.path.exists(DATASET_PATH):
    raise ValueError(f"Директории не существует: {DATASET_PATH}")
if os.path.exists(KG_PATH):
    raise ValueError(f"Директория существует: {KG_PATH}")

os.mkdir(KG_PATH)
os.mkdir(VECTORIZED_DB_PATH)
os.mkdir(GRAPH_DB_PATH)
os.mkdir(CACHE_DB_PATH)

In [4]:
print(VECTORIZED_DB_PATH)
print(GRAPH_DB_PATH)
print(CACHE_DB_PATH)

../../../data/knowledge_graphs/diaasqa/gigachat_full/embeddings_part/
../../../data/knowledge_graphs/diaasqa/gigachat_full/graph_part/
../../../data/knowledge_graphs/diaasqa/gigachat_full/cache_part/


In [5]:
# Setting knowledge graph

graph_config = GraphModelConfig(
    driver_config=GraphDriverConfig(
        db_vendor='neo4j', 
        db_config=GraphDBConnectionConfig(
            uri="bolt://localhost:7687", params={'user': "neo4j", 'pwd': 'password'}, 
            need_to_clear=False)))

embed_config = EmbeddingsModelConfig(
    nodesdb_driver_config=VectorDriverConfig(
        db_vendor='chroma',
        db_config=VectorDBConnectionConfig(
            path=VECTORIZED_DB_PATH, db_info={'db': 'personalaidb', 'table': "vectorized_nodes"}, need_to_clear=False)),
    tripletsdb_driver_config=VectorDriverConfig(
        db_vendor='chroma', 
        db_config=VectorDBConnectionConfig(
            path=VECTORIZED_DB_PATH, db_info={'db': 'personalaidb', 'table': "vectorized_triplets"}, need_to_clear=False)),
    embedder_config=EmbedderModelConfig(model_name_or_path=HYPER_PARAMS['EMBEDDER_MODEL_PATH']))

kg_model = KnowledgeGraphModel(
    graph_config=graph_config,
    embeddings_config=embed_config)

No sentence-transformers model found with name ../../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [6]:
print(kg_model.embeddings_struct.vectordbs['nodes'].count_items())
print(kg_model.embeddings_struct.vectordbs['triplets'].count_items())
print(kg_model.graph_struct.db_conn.count_items())

0
0
{'triplets': 182838, 'nodes': 49597}


In [7]:
# Setting Memorization Pipeline

mem_config = MemPipelineConfig(
    extractor_config=LLMExtractorConfig(),
    updator_config=LLMUpdatorConfig(
        delete_obsolete_info=HYPER_PARAMS['DELETE_OBSOLETE_INFO']))

mem_pipeline = MemPipeline(kg_model, mem_config)

In [8]:
with open(HYPER_PARAMS_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(HYPER_PARAMS, ensure_ascii=False, indent=1))

joblib.dump(graph_config, GRAPH_DRIVER_CONFIG_PATH)
joblib.dump(embed_config, EMBEDDINGS_DRIVER_CONFIG_PATH)
joblib.dump(mem_config, MEM_PIPELINE_CONFIG_PATH)

['../../../data/knowledge_graphs/diaasqa/gigachat_full/mem_pipeline_config']

### Loading dataset

In [9]:
def custom_diaasqa_load(dataset_path: str) -> List[Tuple[str, Dict[str, str]]]:
    with open(dataset_path, 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())

    data_pairs = []
    for item in data['data']:
        data_pairs.append((item['text_dialog'], {'time': item['time'].split(',')[0]}))

    return data_pairs

In [10]:
CUSTOM_LOAD_FUNCS = {
    'diaasqa': custom_diaasqa_load
}

In [11]:
dataset = CUSTOM_LOAD_FUNCS[HYPER_PARAMS['DATASET_NAME']](HYPER_PARAMS['DATASET_PATH'])

In [12]:
print(len(dataset))

3483


### Creating knowledge graph

In [ ]:
saved_triplets = []

In [ ]:
# 2106 | gigachat_full | diaasqa |

In [14]:
for item in tqdm(dataset[2106:]):
    text, properties = item[0], item[1]
    extracted_triplets, _ = mem_pipeline.remember(text, properties)
    saved_triplets.append(extracted_triplets)

100%|██████████| 1377/1377 [8:02:36<00:00, 21.03s/it]


In [ ]:
print(kg_model.embeddings_struct.vectordbs['nodes'].count_items())
print(kg_model.embeddings_struct.vectordbs['triplets'].count_items())
print(kg_model.graph_struct.db_conn.count_items())

### Saving log inforamtion

In [16]:
joblib.dump(saved_triplets, EXTRACTED_TRIPLETS_PATH)

['../../data/knowledge_graphs/diaasqa/gigachat_full/mem_pipeline_config']

### Build graph on extracted triplets

In [8]:
from functools import reduce

In [9]:
saved_triplets = joblib.load(EXTRACTED_TRIPLETS_PATH)

In [10]:
flattened_triplets = reduce(lambda acc, v: acc + v, saved_triplets, [])

In [11]:
output = mem_pipeline.updator.kg_model.graph_struct.create_triplets(flattened_triplets, status_bar=True)

100%|██████████| 2870/2870 [7:25:58<00:00,  9.32s/it]  


{'triplets': {'84995c505b39da014880f3b5c6d5301b',
  '48c1316e0a5818bd15bc33d6be9a31f7',
  'd5e7839047c01f8ec7066c138f86292a',
  '7fa8abd75bdeb4732c714a15fed0e921',
  'bb633d89da1eb8117105417fd192e37d',
  'e436d3762a4c1ecaf8d30a9347b80f31',
  '702d3b075ad84ed00a6406381d80d401',
  '4078896e1eee904aa75b1880a44858e6',
  'af8c847413a473f20337b519fe8ed3c3',
  'ebd2fd8986f03e027410a2497a8fa8d3',
  'e21377caade4a1f881cc1b79b93f78f5',
  '96988be9722bc29dca6c202b9d3784a7',
  '949cd66f7c681f9e96606b5698f98d39',
  '8aacfb1150d6c206003e16fe45c82844',
  '51ac0d2cd03540567460b53ef1e73c74',
  '4d744743287bb715aff8ce6897ff2de3',
  '1aaa08f8b9b1aa52fc8c7c7333292bde',
  'a3f58c4f2afe6e13d0afaf47443ae85b',
  'c611e52ceb71762cf9c8a72985f3d8a1',
  '388406cde198d7133666ee7b1c0f0bee',
  '293777ffdade383971dce76fbc892bcc',
  'b874a1aec289e6281249dba29bbdb349',
  '9c535bb52463321ff9eec2975de92a27',
  '86fd025e81bb918a2d34f01fa59c737b',
  'd38885a30f2b497a86ef0bb189bb18ee',
  'e1a8e679f8ed45efb402f09849be9571',


In [11]:
output = mem_pipeline.updator.kg_model.embeddings_struct.create_triplets(flattened_triplets, status_bar=True)

100%|██████████| 1435/1435 [42:40<00:00,  1.78s/it]


{'nodes': {'7caa1efd661584fc28374e6cd2c03b7a',
  'fb7d35b7dccc78a90ab02bc6f2c0bc59',
  '16c45f63e3365ce9f4b7828bac4611ef',
  'ea3a707ee64bf0731d4b3d4d31866ebe',
  '3e84e278876a7a76ca7361cad22b57b9',
  '9be8b0c7f871a75830a9edba0f4fa59a',
  '1b60c47c24c433f098958d9516699cef',
  '4b03527ce0b691b0ff4c4b078652f6e2',
  'b8d61a779ae261787e040c140c09b0b1',
  'd8a1f386aaa0f8b33dad220bda256747',
  '3808d52626c8743ac72b0fa53920611b',
  'bf7220f20bc3754400f08bdc386cf536',
  '73588641a50a5125e8baf6ec4b1868c4',
  '1d9292ae914bbebd4072907a3aae73af',
  'a9ba42a080b889b86b247801512136b2',
  '5c35c8051956664eb72045fd71bc2ccf',
  '50c7d6449ed9e20e7de9e67be7c409ee',
  '1f691ae50aefaeaea2d63b9e4afc4045',
  '5491a9328b187317e2c3816bd2f46043',
  '3eda44b43d8a82e2a9e581c51686e532',
  '27109273460586e3a6f4e06ed21c2247',
  '9166637675765c4e982879507eb13eec',
  '446c287ada5bb0793aa6ef875fc325bd',
  '2c1b889dfaf5c9862aa5c8672e4d6389',
  'd42bf7dac47eeec61d5fd5723679f7b9',
  'dba1e2e13e8a855537193765b673f4c9',
  '

In [13]:
print(kg_model.embeddings_struct.vectordbs['nodes'].count_items())
print(kg_model.embeddings_struct.vectordbs['triplets'].count_items())
print(kg_model.graph_struct.db_conn.count_items())

49597
44328
{'triplets': 182838, 'nodes': 49597}
